An artificial "genome" is given with the sequence:
ATGATTTCCTCTCCTCGATTCCGCCAATC
We assume the length of the k-mer is k = 6.

**3a. K-mer list**

1) List all k-mers of length 6 found in the genome (starting from position 1).
2) For each k-mer, write:
    - prefix = first 5 nucleotides (k–1),
    - suﬃx = last 5 nucleotides (k–1).

In [1]:
sequence = "ATGATTTCCTCTCCTCGATTCCGCCAATC"
kmer_length = 6

def create_kmer(sequence, k):
    kmers = []
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i + k]
        kmers.append(kmer)
    return kmers

def create_prefix(kmers):
    prefixes = [kmer[:-1] for kmer in kmers]
    return prefixes

def create_suffixes(kmers):
    suffixes = [kmer[1:] for kmer in kmers]
    return suffixes

kmers = create_kmer(sequence, kmer_length)
kmer_prefixes = create_prefix(kmers)
kmer_suffixes = create_suffixes(kmers)

In [2]:
import pandas as pd

unique_nodes = sorted(list(set(kmer_prefixes + kmer_suffixes)))
matrix = pd.DataFrame(0, index=unique_nodes, columns=unique_nodes)

for pref, suff in zip(kmer_prefixes, kmer_suffixes):
    matrix.loc[pref, suff] += 1

in_degree = matrix.sum(axis=0)
out_degree = matrix.sum(axis=1)

in_degree_dict = in_degree.to_dict()
out_degree_dict = out_degree.to_dict()

for node in in_degree_dict:
    if in_degree_dict[node] == 1 and out_degree_dict[node] == 0:
        print(f"Last node: {node}")
    elif in_degree_dict[node] == 0 and out_degree_dict[node] == 1:
        print(f"First node: {node}")

First node: ATGAT
Last node: CAATC


**3b. De Bruijn Graph**

Based on the table in 3a:
1) Build a de Bruijn graph:
    - nodes = all unique 6-mers (prefixes and suffixes),
    - edges = 5-mers (connect prefix → suﬃx).
For each node, mark its in-degree (number of incoming edges) and out-degree (number
outgoing).
In the report show:
    - table (k-mer, prefix, suﬃx),
    - graph diagram (it can be handwritten, it is important that the arrows and the names of the vertices are marked).

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1. Data
genome = "ATGATTTCCTCTCCTCGATTCCGCCAATC"
k = 6
edges = []
for i in range(len(genome) - k + 1):
    kmer = genome[i:i+k]
    edges.append((kmer[:-1], kmer[1:]))

G = nx.DiGraph()
G.add_edges_from(edges)

# 2. Manually defining position for readability (linear layout with loop)
pos = {}

# Node groups
pre_loop = ['ATGAT', 'TGATT', 'GATTT', 'ATTTC', 'TTTCC', 'TTCCT']
hub = 'TCCTC'
loop_nodes = ['CCTCT', 'CTCTC', 'TCTCC', 'CTCCT']
post_loop = ['CCTCG', 'CTCGA', 'TCGAT', 'CGATT', 'GATTC', 'ATTCC', 
             'TTCCG', 'TCCGC', 'CCGCC', 'CGCCA', 'GCCAA', 'CCAAT', 'CAATC']

# Pozycjonowanie
# A. The part before the loop (y=0)
for i, node in enumerate(pre_loop):
    pos[node] = (i, 0)

# B. Nodal node (x=6, y=0)
hub_x = len(pre_loop)
pos[hub] = (hub_x, 0)

# C. Loop (above the junction)
pos['CCTCT'] = (hub_x - 0.5, 1.0)
pos['CTCTC'] = (hub_x - 0.5, 1.5)
pos['TCTCC'] = (hub_x + 0.5, 1.5)
pos['CTCCT'] = (hub_x + 0.5, 1.0)

# D. The part after the loop (y=0)
start_x = hub_x + 1
for i, node in enumerate(post_loop):
    pos[node] = (start_x + i, 0)

# 3. Ploting
plt.figure(figsize=(15, 6))
nx.draw_networkx_nodes(G, pos, node_size=1800, node_color='lightgreen', edgecolors='black')
nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
# connectionstyle='arc3,rad=0.1' adds slight arcs to arrows
nx.draw_networkx_edges(G, pos, edge_color='#555555', arrowsize=15, connectionstyle='arc3,rad=0.1')

plt.axis('off')
plt.tight_layout()
plt.show()

**3c. Contigs in assembly graph**

We want to obtain contigs from the de Bruijn graph, i.e. maximum paths in which everyone is in the middle of the path
the vertex has in-degree = 1 and out-degree = 1 (no branches).
1) Identify vertices that have in-degree ≠ 1 or out-degree ≠ 1 - these are potential origins or
ends of contigs.
2) For each starting node or node with in-degree >1 or out-degree>1, start the path after
graph until you encounter a terminal node or a node with in-degree >1 or out-degree>1. Important:
the condition for completing the path is the presence of at least one node with in-degree=1 and out-degree=1.
3) Save the contig sequence:
    - the first vertex gives the first 5 nucleotides,
    - each subsequent vertex "adds" 1 nucleotide to the end.

List all contigs and their sequences in the report.
Hint (optional Python):
You can write a short script that:
    - generates k-mers,
    - counts in/out-degree,
    - prints contigs as paths in the graph.

**3d. Is the genome clear?**

Based on the obtained graph/contigs:
1) Can you unambiguously reconstruct the entire "genome" (the sequence given at the beginning)?
2) If not, design the shortest sequence (long read) that passes through the appropriate location
in the genome so as to remove ambiguity in the graph.
    - Save the proposed long read sequence.
    - Briefly explain why this sequence solves the problem.